<a href="https://colab.research.google.com/github/emersonheto/books/blob/master/AtividadePratica_Dimensionalidade_Churn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CMP263 - Aprendizagem de Máquina - INF/UFRGS**
#### *Profa. Mariana Recamonde-Mendoza (mrmendoza@inf.ufrgs.br)*
<br>

## Atividade Prática: Redução de Dimensionalidade

---
***Observação:*** *Este notebook é disponibilizado aos alunos como complemento às aulas e aos slides preparados pela professora. Os principais conceitos são apresentados no material teórico. O objetivo deste notebook é reforçar os conceitos e demonstrar questões práticas no uso de diferentes técnicas de redução de dimensionalidade.*

---

**Objetivo deste notebook**: Explorar e comparar técnicas de **redução de dimensionalidade** aplicadas ao dataset de Churn, incluindo:
1. **PCA** (Principal Component Analysis) — técnica de extração de componentes
2. **Seleção de atributos por filtro** (Filter methods) — baseada em estatísticas univariadas
3. **Seleção de atributos embedded** (Embedded methods) — com Random Forest
4. **Seleção de atributos wrapper** (Wrapper methods) — busca sequencial

O pré-processamento é simplificado intencionalmente: usamos **imputação pela mediana** (única estratégia, sem data leakage) e **Random Forest com `class_weight='balanced'`**, que não exige normalização nem re-amostragem explícita.

# 0. Carregamento de Bibliotecas

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

# Redução de dimensionalidade
from sklearn.decomposition import PCA
from sklearn.feature_selection import (
    SelectKBest, chi2, f_classif, mutual_info_classif,
    SelectFromModel, RFE
)


import warnings
warnings.filterwarnings("ignore")
sns.set(style="whitegrid")

SEED = 42
np.random.seed(SEED)

# 1. Carregamento e Preparação Inicial

In [ ]:
# Dataset de churn (Telco Customer Churn)
df = pd.read_csv("https://drive.google.com/uc?export=view&id=10VrzI8mA2wPvkNIDaLzv-IglPU-Febtf")

# Preparação básica
df = df.drop_duplicates(keep='last')
df.drop(['customerID'], axis=1, inplace=True)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

X = df.drop('Churn', axis=1)
y = df['Churn']

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Shape do dataset: {df.shape}")
print(f"Colunas numéricas ({len(num_cols)}): {num_cols}")
print(f"Colunas categóricas ({len(cat_cols)}): {cat_cols}")
print(f"\nDistribuição do alvo:\n{y.value_counts(normalize=True).round(3)}")

In [ ]:
# Divisão treino/teste estratificada — feita ANTES de qualquer transformação
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f"Treino: {X_train_raw.shape[0]} amostras | Teste: {X_test_raw.shape[0]} amostras")
print(f"Distribuição no treino: {y_train.value_counts(normalize=True).round(3).to_dict()}")

# 2. Pré-processamento Base (Fixo para Todos os Experimentos)

Adotamos uma **única estratégia de pré-processamento**, simplificada:

- **Valores ausentes**: imputação pela **mediana** para variáveis numéricas e pela **moda** para categóricas — estimadores ajustados apenas no treino (sem data leakage)
- **Codificação**: `OneHotEncoder` para variáveis categóricas
- **Escalonamento**: `StandardScaler` para variáveis numéricas — **necessário apenas para o PCA**; o Random Forest não o requer, mas o incluímos no pipeline do PCA
- **Desbalanceamento**: tratado diretamente pelo parâmetro `class_weight='balanced'` do Random Forest — sem necessidade de re-amostragem

In [ ]:
# Pré-processador BASE (sem escalonamento) — usado para métodos de seleção de atributos
def make_base_preprocessor():
    num_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median'))
    ])
    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(drop='if_binary', sparse_output=False, handle_unknown='ignore'))
    ])
    return ColumnTransformer([
        ('numerical', num_pipe, num_cols),
        ('categorical', cat_pipe, cat_cols)
    ])

# Pré-processador COM StandardScaler — necessário para PCA
def make_scaled_preprocessor():
    num_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(drop='if_binary', sparse_output=False, handle_unknown='ignore'))
    ])
    return ColumnTransformer([
        ('numerical', num_pipe, num_cols),
        ('categorical', cat_pipe, cat_cols)
    ])

# Modelo base
def make_rf():
    return RandomForestClassifier(n_estimators=50, max_depth=8, class_weight='balanced', random_state=SEED)

# Validação cruzada estratificada
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED) #usando 3 folds para controlar o tempo de execução

print("Pré-processadores definidos com sucesso.")

# 3. Baseline — Sem Redução de Dimensionalidade

Antes de aplicar qualquer técnica de redução, avaliamos o modelo com **todos os atributos** disponíveis após o pré-processamento. Este é o nosso ponto de referência.

In [ ]:
pipe_baseline = Pipeline([
    ('preprocessor', make_base_preprocessor()),
    ('classifier', make_rf())
])

scores_baseline_f1 = cross_val_score(pipe_baseline, X_train_raw, y_train,
                                      scoring='f1', cv=cv, n_jobs=-1)
scores_baseline_roc = cross_val_score(pipe_baseline, X_train_raw, y_train,
                                       scoring='roc_auc', cv=cv, n_jobs=-1)

print(f"[Baseline — todos os atributos]")
print(f"  F1 médio:      {scores_baseline_f1.mean():.4f} ± {scores_baseline_f1.std():.4f}")
print(f"  ROC-AUC médio: {scores_baseline_roc.mean():.4f} ± {scores_baseline_roc.std():.4f}")

# Quantas features o preprocessador gera?
pipe_baseline.fit(X_train_raw, y_train)
n_features_total = pipe_baseline.named_steps['preprocessor'].transform(X_train_raw).shape[1]
print(f"\nTotal de features após one-hot encoding: {n_features_total}")

# 4. Parte A — PCA (Principal Component Analysis)

O PCA é uma técnica de **extração de características**: os componentes principais são combinações lineares de todos os atributos originais. O espaço original é projetado em direções de máxima variância.

**Importante**: PCA é sensível à escala, por isso usamos o `StandardScaler` antes de aplicá-lo.

## 4.1 Análise da variância explicada

In [ ]:
# Ajustar o pré-processador e aplicar PCA apenas para análise exploratória
preprocessor_scaled = make_scaled_preprocessor()
X_train_scaled = preprocessor_scaled.fit_transform(X_train_raw, y_train)

pca_full = PCA(random_state=SEED)
pca_full.fit(X_train_scaled)

variancia_acumulada = np.cumsum(pca_full.explained_variance_ratio_)

# Número de componentes para atingir limiares de variância
for limiar in [0.80, 0.90, 0.95, 0.99]:
    n = np.argmax(variancia_acumulada >= limiar) + 1
    print(f"  {limiar*100:.0f}% da variância explicada com {n} componentes")

# Gráfico de variância acumulada
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(range(1, len(pca_full.explained_variance_ratio_) + 1),
            pca_full.explained_variance_ratio_, color='steelblue', alpha=0.7)
axes[0].set_xlabel('Componente Principal')
axes[0].set_ylabel('Variância Explicada')
axes[0].set_title('Variância Explicada por Componente (Scree Plot)')
axes[0].set_xlim(0.5, 30)

axes[1].plot(range(1, len(variancia_acumulada) + 1), variancia_acumulada,
             color='steelblue', marker='o', markersize=3)
for limiar, cor in [(0.80, 'green'), (0.90, 'orange'), (0.95, 'red')]:
    n = np.argmax(variancia_acumulada >= limiar) + 1
    axes[1].axhline(y=limiar, color=cor, linestyle='--', alpha=0.7,
                    label=f'{limiar*100:.0f}% ({n} comp.)')
    axes[1].axvline(x=n, color=cor, linestyle='--', alpha=0.4)
axes[1].set_xlabel('Número de Componentes')
axes[1].set_ylabel('Variância Acumulada')
axes[1].set_title('Variância Acumulada vs Número de Componentes')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4.2 Comparação de diferentes números de componentes

In [ ]:
resultados_pca = []

# Limiares de variância a testar
n_componentes_list = [5, 10, 15, 20]
# Adiciona os limiares de variância como opções de n_components
for var_limiar in [0.80, 0.90, 0.95]:
    n = int(np.argmax(variancia_acumulada >= var_limiar) + 1)
    if n not in n_componentes_list:
        n_componentes_list.append(n)
n_componentes_list = sorted(set(n_componentes_list))

for n_comp in n_componentes_list:
    pipe_pca = Pipeline([
        ('preprocessor', make_scaled_preprocessor()),
        ('pca', PCA(n_components=n_comp, random_state=SEED)),
        ('classifier', make_rf())
    ])
    f1_scores = cross_val_score(pipe_pca, X_train_raw, y_train,
                                scoring='f1', cv=cv, n_jobs=-1)
    roc_scores = cross_val_score(pipe_pca, X_train_raw, y_train,
                                 scoring='roc_auc', cv=cv, n_jobs=-1)
    var_exp = variancia_acumulada[n_comp - 1]
    resultados_pca.append({
        'n_componentes': n_comp,
        'var_acumulada': round(var_exp * 100, 1),
        'F1_mean': round(f1_scores.mean(), 4),
        'F1_std': round(f1_scores.std(), 4),
        'ROC_mean': round(roc_scores.mean(), 4),
        'ROC_std': round(roc_scores.std(), 4)
    })
    print(f"  PCA(n={n_comp:2d}, var={var_exp*100:.1f}%) — F1: {f1_scores.mean():.4f} | ROC-AUC: {roc_scores.mean():.4f}")

df_pca = pd.DataFrame(resultados_pca)
print("\n", df_pca.to_string(index=False))

In [ ]:
# Visualização: F1 x número de componentes
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_pca['n_componentes'], df_pca['F1_mean'], marker='o', color='steelblue', label='PCA')
ax.fill_between(df_pca['n_componentes'],
                df_pca['F1_mean'] - df_pca['F1_std'],
                df_pca['F1_mean'] + df_pca['F1_std'], alpha=0.2, color='steelblue')
ax.axhline(scores_baseline_f1.mean(), color='red', linestyle='--',
           label=f'Baseline (todos os atributos): {scores_baseline_f1.mean():.4f}')
ax.set_xlabel('Número de Componentes Principais')
ax.set_ylabel('F1-Score (CV=5)')
ax.set_title('Desempenho do PCA em função do número de componentes')
ax.legend()
plt.tight_layout()
plt.show()

## 4.3 Interpretação dos Componentes Principais

Veja as contribuições das features originais para o PC1 e PC2.

In [ ]:
# Obter nomes das features após o pré-processamento
preprocessor_scaled_fit = make_scaled_preprocessor()
preprocessor_scaled_fit.fit(X_train_raw)

feature_names_num = num_cols
feature_names_cat = preprocessor_scaled_fit.named_transformers_['categorical']['encoder'].get_feature_names_out(cat_cols).tolist()
all_feature_names = feature_names_num + feature_names_cat

# Loadings do PCA
pca_vis = PCA(n_components=5, random_state=SEED)
X_train_pca = preprocessor_scaled_fit.transform(X_train_raw)
pca_vis.fit(X_train_pca)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for idx, ax in enumerate(axes):
    loadings = pca_vis.components_[idx]
    sorted_idx = np.argsort(np.abs(loadings))[::-1][:15]
    ax.barh([all_feature_names[i] for i in sorted_idx[::-1]],
            [loadings[i] for i in sorted_idx[::-1]],
            color=['steelblue' if v > 0 else 'salmon' for v in [loadings[i] for i in sorted_idx[::-1]]])
    ax.set_title(f'PC{idx+1} — Top 15 contribuições\n(var. explicada: {pca_vis.explained_variance_ratio_[idx]*100:.1f}%)')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Loading')
plt.tight_layout()
plt.show()

print("\n💡 Observação: PC1 e PC2 são combinações lineares de TODOS os atributos originais.")
print("   Não é possível dizer 'o PC1 representa a variável X' — ele mistura múltiplas variáveis.")
print("   Isso é a principal desvantagem do PCA em termos de interpretabilidade.")

# 5. Parte B — Seleção de Atributos por Filtro (Filter Methods)

Os métodos de filtro **selecionam atributos individualmente**, com base em métricas estatísticas calculadas entre cada feature e o alvo — **independentemente do modelo**. São rápidos e computacionalmente baratos.

| Critério | Tipo de feature | Característica |
|---|---|---|
| `f_classif` (ANOVA F) | Numérica | Relação linear entre feature e alvo |
| `mutual_info_classif` | Qualquer | Dependência estatística (inclui não-linear) |
| `chi2` | Categórica (não-negativa) | Independência entre feature categórica e alvo |

In [ ]:
# Pré-processar os dados uma vez para análise exploratória dos filtros
pre = make_base_preprocessor()
X_train_proc = pre.fit_transform(X_train_raw)
n_features_total = X_train_proc.shape[1]

# Calcular scores de cada critério
selector_f = SelectKBest(score_func=f_classif, k='all')
selector_f.fit(X_train_proc, y_train)

selector_mi = SelectKBest(score_func=mutual_info_classif, k='all')
selector_mi.fit(X_train_proc, y_train)

# Nomes das features
enc = pre.named_transformers_['categorical']['encoder']
feat_names_cat = enc.get_feature_names_out(cat_cols).tolist()
feat_names = num_cols + feat_names_cat

df_scores = pd.DataFrame({
    'feature': feat_names,
    'f_classif': selector_f.scores_,
    'mutual_info': selector_mi.scores_
}).sort_values('mutual_info', ascending=False)

# Top 15 features por cada critério
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, col, titulo in zip(axes, ['f_classif', 'mutual_info'],
                            ['ANOVA F-value (f_classif)', 'Informação Mútua']):
    top = df_scores.nlargest(15, col)
    ax.barh(top['feature'][::-1], top[col][::-1], color='steelblue', alpha=0.8)
    ax.set_title(f'Top 15 features — {titulo}')
    ax.set_xlabel('Score')
plt.tight_layout()
plt.show()

print("Top 10 por Informação Mútua:")
print(df_scores[['feature', 'f_classif', 'mutual_info']].head(10).to_string(index=False))

In [ ]:
# Comparação de desempenho com diferentes k e critérios
resultados_filter = []
k_values = [5, 10, 15, 20, 25, n_features_total]

for score_func, nome in [(f_classif, 'f_classif'), (mutual_info_classif, 'mutual_info')]:
    for k in k_values:
        k_real = min(k, n_features_total)
        pipe = Pipeline([
            ('preprocessor', make_base_preprocessor()),
            ('selector', SelectKBest(score_func=score_func, k=k_real)),
            ('classifier', make_rf())
        ])
        f1 = cross_val_score(pipe, X_train_raw, y_train, scoring='f1', cv=cv, n_jobs=-1)
        roc = cross_val_score(pipe, X_train_raw, y_train, scoring='roc_auc', cv=cv, n_jobs=-1)
        resultados_filter.append({
            'metodo': nome, 'k': k_real,
            'F1_mean': f1.mean(), 'F1_std': f1.std(),
            'ROC_mean': roc.mean(), 'ROC_std': roc.std()
        })
        print(f"  [{nome}, k={k_real:2d}] F1: {f1.mean():.4f} | ROC-AUC: {roc.mean():.4f}")

df_filter = pd.DataFrame(resultados_filter)

In [ ]:
# Gráfico comparativo dos métodos de filtro
fig, ax = plt.subplots(figsize=(10, 4))
for metodo, cor in [('f_classif', 'steelblue'), ('mutual_info', 'darkorange')]:
    sub = df_filter[df_filter['metodo'] == metodo]
    ax.plot(sub['k'], sub['F1_mean'], marker='o', color=cor, label=metodo)
    ax.fill_between(sub['k'], sub['F1_mean'] - sub['F1_std'],
                    sub['F1_mean'] + sub['F1_std'], alpha=0.15, color=cor)
ax.axhline(scores_baseline_f1.mean(), color='red', linestyle='--',
           label=f'Baseline: {scores_baseline_f1.mean():.4f}')
ax.set_xlabel('Número de features selecionadas (k)')
ax.set_ylabel('F1-Score (CV=5)')
ax.set_title('Filter Methods: desempenho por número de features')
ax.legend()
plt.tight_layout()
plt.show()

# 6. Parte C — Seleção Embedded com Random Forest

Nos métodos embedded, a seleção de atributos ocorre **durante o treinamento do próprio modelo**. O Random Forest calcula a importância de cada feature como parte do processo de construção das árvores (impureza de Gini ou permutação).

`SelectFromModel` usa um limiar sobre as importâncias para selecionar features.

In [ ]:
# Visualizar importâncias do Random Forest treinado no conjunto de treino completo
pre_fit = make_base_preprocessor()
X_train_proc_all = pre_fit.fit_transform(X_train_raw)

enc_fit = pre_fit.named_transformers_['categorical']['encoder']
feat_names_all = num_cols + enc_fit.get_feature_names_out(cat_cols).tolist()

rf_fit = make_rf()
rf_fit.fit(X_train_proc_all, y_train)

importancias = pd.Series(rf_fit.feature_importances_, index=feat_names_all)
importancias_sorted = importancias.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
importancias_sorted.head(20).plot(kind='bar', ax=ax, color='steelblue', alpha=0.8)
ax.set_title('Top 20 features — Importância pelo Random Forest (Gini)')
ax.set_ylabel('Importância')
ax.set_xlabel('Feature')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f"\nTop 10 features mais importantes:")
print(importancias_sorted.head(10).round(4).to_string())

In [ ]:
# SelectFromModel com diferentes limiares
resultados_embedded = []

for limiar in ['mean', 'median', '1.25*mean', '1.5*mean']:
    pipe = Pipeline([
        ('preprocessor', make_base_preprocessor()),
        ('selector', SelectFromModel(make_rf(), threshold=limiar)),
        ('classifier', make_rf())
    ])
    # Descobrir quantas features são selecionadas
    pipe.fit(X_train_raw, y_train)
    n_sel = pipe.named_steps['selector'].get_support().sum()

    f1 = cross_val_score(pipe, X_train_raw, y_train, scoring='f1', cv=cv, n_jobs=-1)
    roc = cross_val_score(pipe, X_train_raw, y_train, scoring='roc_auc', cv=cv, n_jobs=-1)
    resultados_embedded.append({
        'limiar': limiar, 'n_features': n_sel,
        'F1_mean': f1.mean(), 'F1_std': f1.std(),
        'ROC_mean': roc.mean(), 'ROC_std': roc.std()
    })
    print(f"  [Embedded, limiar={limiar:10s}, n={n_sel:2d}] F1: {f1.mean():.4f} | ROC-AUC: {roc.mean():.4f}")

df_embedded = pd.DataFrame(resultados_embedded)
print("\n", df_embedded[['limiar','n_features','F1_mean','F1_std','ROC_mean','ROC_std']].to_string(index=False))

# 7. Parte D — Seleção Wrapper: Recursive Feature Elimination (RFE)

Os métodos wrapper **avaliam subconjuntos de features** treinando e avaliando o modelo para cada subconjunto candidato. São mais custosos computacionalmente, mas tendem a encontrar subconjuntos mais adequados ao modelo específico.

RFE (Recursive Feature Elimination): seleciona os atributos mais relevantes eliminando iterativamente aqueles menos importantes, com base no desempenho de um modelo.

In [ ]:
# Parte D — Wrapper: RFE (Recursive Feature Elimination)
# RFE dentro do CV sem viés metodológico.
# Estimador interno: make_rf() — mesmo modelo usado em todo o notebook.



k_values_rfe = [5, 10, 15, 20]   # mesmos k dos filtros
resultados_wrapper = []

for n_feat in k_values_rfe:
    pipe_rfe = Pipeline([
        ('preprocessor', make_base_preprocessor()),
        ('selector',     RFE(estimator=make_rf(),
                             n_features_to_select=n_feat,
                             step=1)),
        ('classifier',   make_rf())
    ])

    f1  = cross_val_score(pipe_rfe, X_train_raw, y_train,
                          scoring='f1',      cv=cv, n_jobs=-1)
    roc = cross_val_score(pipe_rfe, X_train_raw, y_train,
                          scoring='roc_auc', cv=cv, n_jobs=-1)

    resultados_wrapper.append({
        'metodo':     f'RFE (n={n_feat})',
        'n_features':  n_feat,
        'F1_mean':    f1.mean(), 'F1_std':  f1.std(),
        'ROC_mean':  roc.mean(), 'ROC_std': roc.std()
    })
    print(f"  [RFE, n={n_feat:2d}] F1: {f1.mean():.4f} ± {f1.std():.4f} | ROC-AUC: {roc.mean():.4f}")

df_wrapper = pd.DataFrame(resultados_wrapper)

# Curva: F1 por número de features
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_wrapper['n_features'], df_wrapper['F1_mean'],
        marker='o', color='#9C27B0', label='RFE')
ax.fill_between(df_wrapper['n_features'],
                df_wrapper['F1_mean'] - df_wrapper['F1_std'],
                df_wrapper['F1_mean'] + df_wrapper['F1_std'],
                alpha=0.15, color='#9C27B0')
ax.axhline(scores_baseline_f1.mean(), color='gray', linestyle=':',
           label=f'Baseline: {scores_baseline_f1.mean():.4f}')
ax.set_xlabel('Número de features selecionadas (k)')
ax.set_ylabel('F1-Score (CV=5)')
ax.set_title('RFE — F1 por número de features')
ax.set_xticks(k_values_rfe)
ax.legend()
plt.tight_layout()
plt.show()

# Features selecionadas pelo melhor n
melhor_n = df_wrapper.loc[df_wrapper['F1_mean'].idxmax(), 'n_features']
pipe_rfe_best = Pipeline([
    ('preprocessor', make_base_preprocessor()),
    ('selector',     RFE(estimator=make_rf(),
                         n_features_to_select=melhor_n, step=1))
])
pipe_rfe_best.fit(X_train_raw, y_train)
enc_rfe = pipe_rfe_best.named_steps['preprocessor'] \
              .named_transformers_['categorical']['encoder']
feat_names_rfe = num_cols + enc_rfe.get_feature_names_out(cat_cols).tolist()
features_selecionadas = [f for f, s in
                         zip(feat_names_rfe,
                             pipe_rfe_best.named_steps['selector'].support_)
                         if s]
print(f"\nMelhor RFE: n={melhor_n} features")
print(f"Features selecionadas: {features_selecionadas}")

print("\n💡 O RFE usa make_rf() para calcular importâncias e guiar a eliminação,")
print("   o mesmo modelo usado como classificador final — configuração consistente")
print("   em todo o notebook (50 árvores, max_depth=8). O RFE é re-ajustado dentro")
print("   de cada fold do CV, sem viés metodológico.")

# 8. Comparação Final — Todos os Métodos

Reunimos os melhores resultados de cada abordagem para uma comparação direta.

In [ ]:
# Selecionar o melhor resultado de cada método
melhor_pca      = df_pca.loc[df_pca['F1_mean'].idxmax()]
melhor_filter_f = df_filter[df_filter['metodo']=='f_classif'].loc[
                      df_filter[df_filter['metodo']=='f_classif']['F1_mean'].idxmax()]
melhor_filter_mi = df_filter[df_filter['metodo']=='mutual_info'].loc[
                      df_filter[df_filter['metodo']=='mutual_info']['F1_mean'].idxmax()]
melhor_embedded  = df_embedded.loc[df_embedded['F1_mean'].idxmax()]
melhor_wrapper   = df_wrapper.loc[df_wrapper['F1_mean'].idxmax()]

tabela_final = pd.DataFrame([
    {'Método': 'Baseline (todos)',
     'n_features': n_features_total,
     'F1': scores_baseline_f1.mean(), 'F1_std': scores_baseline_f1.std(),
     'ROC': scores_baseline_roc.mean()},
    {'Método': f'PCA (n={int(melhor_pca["n_componentes"])})',
     'n_features': int(melhor_pca['n_componentes']),
     'F1': melhor_pca['F1_mean'], 'F1_std': melhor_pca['F1_std'],
     'ROC': melhor_pca['ROC_mean']},
    {'Método': f'Filter f_classif (k={int(melhor_filter_f["k"])})',
     'n_features': int(melhor_filter_f['k']),
     'F1': melhor_filter_f['F1_mean'], 'F1_std': melhor_filter_f['F1_std'],
     'ROC': melhor_filter_f['ROC_mean']},
    {'Método': f'Filter MutInfo (k={int(melhor_filter_mi["k"])})',
     'n_features': int(melhor_filter_mi['k']),
     'F1': melhor_filter_mi['F1_mean'], 'F1_std': melhor_filter_mi['F1_std'],
     'ROC': melhor_filter_mi['ROC_mean']},
    {'Método': f'Embedded RF (limiar={melhor_embedded["limiar"]})',
     'n_features': int(melhor_embedded['n_features']),
     'F1': melhor_embedded['F1_mean'], 'F1_std': melhor_embedded['F1_std'],
     'ROC': melhor_embedded['ROC_mean']},
    {'Método': melhor_wrapper['metodo'],
     'n_features': int(melhor_wrapper['n_features']),
     'F1': melhor_wrapper['F1_mean'], 'F1_std': melhor_wrapper['F1_std'],
     'ROC': melhor_wrapper['ROC_mean']},
])

print("=== Comparação Final — Melhores Resultados por Método ===")
print(tabela_final.round(4).to_string(index=False))

# Gráfico de barras comparativo
fig, ax = plt.subplots(figsize=(12, 5))
cores = ['#9E9E9E', '#2196F3', '#FF9800', '#FF5722', '#4CAF50', '#9C27B0']
bars = ax.bar(tabela_final['Método'], tabela_final['F1'], color=cores,
              yerr=tabela_final['F1_std'], capsize=4, alpha=0.85)
ax.set_ylabel('F1-Score (CV=5)')
ax.set_title('Comparação de Métodos de Redução de Dimensionalidade')
ax.set_ylim(tabela_final['F1'].min() * 0.95, tabela_final['F1'].max() * 1.05)
plt.xticks(rotation=20, ha='right')
for bar, val in zip(bars, tabela_final['F1']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

# 9. Avaliação Final no Conjunto de Teste

Selecionamos o **melhor pipeline** com base na validação cruzada e no F1, e avaliamos no conjunto de teste holdout — que **não foi usado em nenhuma etapa anterior**.

In [ ]:
# ── Melhor método: Filter por Informação Mútua ──
melhor_k_mi = df_filter[df_filter['metodo']=='mutual_info'].loc[
                  df_filter[df_filter['metodo']=='mutual_info']['F1_mean'].idxmax(), 'k']

best_pipeline = Pipeline([
    ('preprocessor', make_base_preprocessor()),
    ('selector', SelectKBest(score_func=mutual_info_classif, k=int(melhor_k_mi))),
    ('classifier', make_rf())
])

best_pipeline.fit(X_train_raw, y_train)
y_pred = best_pipeline.predict(X_test_raw)

n_feat_test = best_pipeline.named_steps['selector'].get_support().sum()
print(f"Features selecionadas pelo melhor pipeline (Filter MI, k={int(melhor_k_mi)}): {n_feat_test}")
print("\n=== Relatório de Classificação — Conjunto de Teste ===")
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=['No Churn', 'Churn'],
    cmap='Blues', ax=ax
)
ax.set_title('Matriz de Confusão — Conjunto de Teste')
plt.tight_layout()
plt.show()